# Axis 1: Representation Probing (Simple Approach)

This notebook probes **final embeddings** from the DreaMS model to understand what chemical information is encoded.

## Goals
1. Probe for physicochemical properties (MW, LogP, TPSA)
2. Probe for functional groups (aromatic, hydroxyl, etc.)
3. Compare linear vs MLP probes
4. Validate with kNN retrieval and UMAP

## Later: Expand to per-layer analysis to see WHERE information emerges

## Setup

In [2]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from simple_probing import SimpleProbe, knn_validation, umap_visualization

print("✅ Imports successful")

✅ Imports successful


## Step 1: Load Final Embeddings

**TODO**: Get final embeddings from DreaMS model.

Two options:
1. **If you have saved embeddings**: Load from file
2. **If you need to extract**: Run DreaMS model and save final layer output

### Option 1: Load saved embeddings

In [3]:
# Option 1: Load pre-saved embeddings
# embeddings = np.load('../data/embeddings/final_embeddings.npy')
# print(f"Loaded embeddings: {embeddings.shape}")

# Option 2: Extract from model (example - adjust for your setup)
# from your_dreams_model import load_model, extract_embeddings
# model = load_model('path/to/checkpoint')
# embeddings = extract_embeddings(model, dataloader)
# np.save('../data/embeddings/final_embeddings.npy', embeddings)

print("⚠️ TODO: Load or extract final embeddings")
# For now, create dummy data for testing
embeddings = np.random.randn(1000, 1024)  # Replace with actual embeddings!

⚠️ TODO: Load or extract final embeddings


## Step 2: Load Targets from Enriched Dataset

In [5]:
# Load enriched dataset
df = pd.read_csv('../data/processed/MassSpecGym_enriched.tsv', sep='\t')

# Make sure we have same number of samples as embeddings
#assert len(df) == len(embeddings), "Mismatch between embeddings and dataset!"

print(f"Dataset shape: {df.shape}")
print(f"Embeddings shape: {embeddings.shape}")

Dataset shape: (231104, 34)
Embeddings shape: (1000, 1024)


## Step 3: Prepare Targets

### 3.1 Physicochemical Properties (Regression)

In [6]:
# Regression targets
targets_regression = {
    'mol_weight': df['mol_weight'].values,
    'logp': df['logp'].values,
    'tpsa': df['tpsa'].values
}

print("Regression targets:")
for name, values in targets_regression.items():
    print(f"  {name}: {values.min():.1f} - {values.max():.1f}, mean={values.mean():.1f}")

Regression targets:
  mol_weight: 59.1 - 998.9, mean=396.2
  logp: -13.1 - 17.9, mean=2.8
  tpsa: 0.0 - 476.5, mean=97.4


### 3.2 Functional Groups (Binary Classification)

In [7]:
# Binary targets - functional groups
functional_groups = ['aromatic', 'hydroxyl', 'ketone', 'amine_primary', 'ester']

targets_binary = {}
for fg in functional_groups:
    if fg in df.columns:
        targets_binary[fg] = df[fg].values.astype(float)
        prevalence = targets_binary[fg].mean()
        print(f"{fg}: {prevalence:.1%} positive samples")

aromatic: 72.0% positive samples
hydroxyl: 55.1% positive samples
ketone: 62.1% positive samples
amine_primary: 12.6% positive samples
ester: 24.0% positive samples


## Step 4: Run Probing Experiments

### 4.1 Initialize Prober

In [8]:
# Create probe object
prober = SimpleProbe(embeddings, random_state=42)
print("✅ Prober initialized")

✅ Prober initialized


### 4.2 Test Individual Targets (Quick Check)

In [9]:
# Quick test: aromatic group with linear probe
prober.probe_binary(
    target=targets_binary['aromatic'],
    target_name='aromatic',
    probe_type='linear'
)

ValueError: Found input variables with inconsistent numbers of samples: [1000, 231104]

In [ ]:
# Quick test: molecular weight with linear probe
prober.probe_regression(
    target=targets_regression['mol_weight'],
    target_name='mol_weight',
    probe_type='linear'
)

### 4.3 Compare Linear vs MLP Across All Targets

In [ ]:
# Run comprehensive comparison
prober.compare_linear_vs_mlp(
    targets_binary=targets_binary,
    targets_regression=targets_regression
)

### 4.4 Visualize Results

In [ ]:
# Plot comparison
prober.plot_results(save_path='../results/linear_vs_mlp.png')

## Step 5: Validation - kNN Retrieval

In [ ]:
# Check if similar molecules (by property) are close in embedding space
print("kNN Validation:\n")

# For aromatic group
knn_metrics = knn_validation(
    embeddings=embeddings,
    labels=targets_binary['aromatic'],
    k=10
)
print(f"Aromatic - Precision@10: {knn_metrics['precision@10']:.3f}")

# For hydroxyl group
knn_metrics = knn_validation(
    embeddings=embeddings,
    labels=targets_binary['hydroxyl'],
    k=10
)
print(f"Hydroxyl - Precision@10: {knn_metrics['precision@10']:.3f}")

## Step 6: UMAP Visualization

In [ ]:
# Visualize embedding space colored by aromatic groups
umap_visualization(
    embeddings=embeddings,
    labels=targets_binary['aromatic'],
    label_name='Aromatic',
    save_path='../results/umap_aromatic.png'
)

In [ ]:
# Visualize embedding space colored by molecular weight
umap_visualization(
    embeddings=embeddings,
    labels=targets_regression['mol_weight'],
    label_name='Molecular Weight',
    save_path='../results/umap_molweight.png'
)

## Summary & Insights

### Key Questions to Answer:

1. **What's encoded in final embeddings?**
   - Which properties have high probe accuracy?
   - Are functional groups linearly separable?

2. **Linear vs MLP?**
   - Where do MLPs significantly outperform linear probes?
   - Evidence of non-linear feature combinations?

3. **Validation consistent?**
   - Do kNN results align with probe findings?
   - Clear clustering in UMAP?

### Next Steps:
- If this works well → Expand to **per-layer analysis** to see WHERE information emerges
- Investigate which properties are hardest to probe
- Test on different functional groups
- Compare frozen vs fine-tuned embeddings

In [ ]:
# Print final summary of results
print("\n" + "="*60)
print("PROBING RESULTS SUMMARY")
print("="*60)

for task_name, metrics in prober.results.items():
    print(f"\n{task_name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.3f}")